# Lab 6C: Voice Cloning with SpeechT5 + Speaker Embeddings

### AIML ZG527: Audio Analysis
**Authored by:** Seetha Parameswaran, BITS Pilani WILP


### Learning Objectives
- Extract speaker embeddings from reference audio using SpeechBrain x-vector
- Clone voice via SpeechT5 conditioned on speaker embedding
- Compare cloned output across two different reference speakers
---

### Setup and Import Libraries
```
!pip install -q transformers soundfile datasets "speechbrain==0.5.16" --break-system-packages
```

---
### Reference Speaker Audio
Using Librispeech speakers:

```
!wget -O librispeech-dev-clean.tar.gz https://www.openslr.org/resources/12/dev-clean.tar.gz
!tar -xzf librispeech-dev-clean.tar.gz
```

In [1]:
!pip install -q transformers soundfile datasets "speechbrain==0.5.16" --break-system-packages

In [2]:
import IPython.display as ipd
import librosa, librosa.display
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
from num2words import num2words
import re

In [3]:
from speechbrain.pretrained import EncoderClassifier
import torchaudio

classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb")

signal, fs = torchaudio.load("/home/seetha/PythonScriptsCourses/PythonScriptsCourses/Audio Analysis/Data/LibriSpeech/dev-clean/1272/128104/1272-128104-0000.flac")
speaker_embeddings = classifier.encode_batch(signal).squeeze(0)

/home/seetha/PythonScriptsCourses/PythonScriptsCourses/lib/python3.8/site-packages/speechbrain/utils/checkpoints.py:147: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.

In [4]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

2026-07-03 09:07:59.649235: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-07-03 09:07:59.649286: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [5]:
TARGET_TEXT = "This voice has been cloned from a short reference recording."
inputs = processor(text=TARGET_TEXT, return_tensors="pt")
speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)
ipd.display(ipd.Audio(speech.numpy(), rate=16000))

In [7]:
# Play reference vs cloned output
REFERENCE_SPEAKER_WAV_1 = "/home/seetha/PythonScriptsCourses/PythonScriptsCourses/Audio Analysis/Data/LibriSpeech/dev-clean/1993/147149/1993-147149-0000.flac"   # 5-10 sec clean speech sample

ref_audio1, ref_sr1 = librosa.load(REFERENCE_SPEAKER_WAV_1, sr=None)
print("Reference speaker 1:")
ipd.display(ipd.Audio(ref_audio1, rate=ref_sr1))

signal1, fs1 = torchaudio.load(REFERENCE_SPEAKER_WAV_1)
speaker_embeddings1 = classifier.encode_batch(signal1).squeeze(0)

print("Cloned voice, new text:")
inputs = processor(text=TARGET_TEXT, return_tensors="pt")
speech1 = model.generate_speech(inputs["input_ids"], speaker_embeddings1, vocoder=vocoder)

ipd.display(ipd.Audio(speech1.numpy(), rate=16000))

Reference speaker 1:


Cloned voice, new text:


In [8]:
# Clone the same text into a second, different speaker for comparison
REFERENCE_SPEAKER_WAV_2 = "/home/seetha/PythonScriptsCourses/PythonScriptsCourses/Audio Analysis/Data/LibriSpeech/dev-clean/1673/143396/1673-143396-0000.flac"

ref_audio2, ref_sr2 = librosa.load(REFERENCE_SPEAKER_WAV_2, sr=None)
print("Reference speaker 2:")
ipd.display(ipd.Audio(ref_audio2, rate=ref_sr2))

signal2, fs2 = torchaudio.load(REFERENCE_SPEAKER_WAV_2)
speaker_embeddings2 = classifier.encode_batch(signal2).squeeze(0)

inputs = processor(text=TARGET_TEXT, return_tensors="pt")
speech2 = model.generate_speech(inputs["input_ids"], speaker_embeddings2, vocoder=vocoder)
print("Same text, second cloned speaker:")
ipd.display(ipd.Audio(speech2.numpy(), rate=16000))

Reference speaker 2:


Same text, second cloned speaker:


## Summary

In this lab, you learned:

1. **Speaker Embedding Extraction**: x-vector via SpeechBrain, from raw reference audio
2. **Voice Cloning**: SpeechT5 conditioned on speaker embedding generates new text in that voice
3. **Multi-Speaker Comparison**: Same text synthesized in two distinct cloned voices

### Key Takeaways
- TTS backbone — voice identity comes entirely from the embedding, not model retraining
- Quality depends on reference clip length/cleanliness — ties to Speaker Embeddings slide (5-10 sec minimum)
- Always use voices with explicit consent — ties to Voice Cloning Risks slide